# Model Development

In this notebook, the preprocessed dataset is used to train and optimize machine learning models for intent classification. The workflow includes data loading, train-validation splitting, model training, hyperparameter tuning, and selecting the best-performing pipeline for final evaluation.

In [ ]:
import pandas as pd
df = pd.read_parquet("../data/preprocessed.parquet")

In [ ]:
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 100)

df.head()

,text,label,label_text,word_count,char_count,utterance_type,tokens,tokens_no_stop,tokens_clean,lemmatized,processed_text
0,I am still waiting on my card?,11,card_arrival,7,30,Question,"[i, am, still, waiting, on, my, card]","[still, waiting, card]","[still, waiting, card]","[still, wait, card]",still wait card
1,What can I do if my card still hasn't arrived after 2 weeks?,11,card_arrival,13,60,Question,"[what, can, i, do, if, my, card, still, has, not, arrived, after, 2, weeks]","[card, still, not, arrived, 2, weeks]","[card, still, not, arrived, 2, weeks]","[card, still, not, arrive, 2, week]",card still not arrive 2 week
2,I have been waiting over a week. Is the card still coming?,11,card_arrival,12,58,Question,"[i, have, been, waiting, over, a, week, is, the, card, still, coming]","[waiting, week, card, still, coming]","[waiting, week, card, still, coming]","[wait, week, card, still, come]",wait week card still come
3,Can I track my card while it is in the process of delivery?,11,card_arrival,13,59,Question,"[can, i, track, my, card, while, it, is, in, the, process, of, delivery]","[track, card, process, delivery]","[track, card, process, delivery]","[track, card, process, delivery]",track card process delivery
4,"How do I know if I will get my card, or if it is lost?",11,card_arrival,15,54,Question,"[how, do, i, know, if, i, will, get, my, card, or, if, it, is, lost]","[know, get, card, lost]","[know, get, card, lost]","[know, get, card, lose]",know get card lose


## Create and Save the Test Set

To ensure a fair and reproducible evaluation, the dataset is split into training and test sets using stratified sampling to preserve the original class distribution.

Only the test subset is saved as a separate file, allowing the final evaluation to be performed independently without repeating the data splitting process.

In [ ]:
from sklearn.model_selection import train_test_split

X = df["processed_text"]
y = df["label_text"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

test_df = pd.DataFrame({
    "processed_text":X_test,
    "label_text":y_test
})

test_df.to_parquet("../data/test.parquet", index=False)

## Model Selection and Hyperparameter Optimization

This stage compares three classical machine learning classifiers for intent classification:

- **Multinomial Naive Bayes**
- **Logistic Regression**
- **LinearSVC**

Each classifier is paired with its corresponding text vectorization method through a Scikit-learn `Pipeline`. **CountVectorizer** is evaluated with Multinomial Naive Bayes, while **TF-IDF Vectorizer** is used with Logistic Regression and LinearSVC. This pipeline-based approach ensures that feature extraction is performed independently within each cross-validation fold, preventing data leakage.

### Vectorizer and Hyperparameter Selection

Rather than using fixed vectorizer settings, **GridSearchCV** jointly optimizes both the vectorizer and classifier hyperparameters. The search explores different vocabulary filtering thresholds (`min_df` and `max_df`), n-gram configurations (for CountVectorizer), and model-specific parameters such as regularization strength (`C`), optimization solver, loss function, and smoothing (`alpha`).

This joint optimization allows the model to identify the most effective combination of text representation and classifier for the Banking77 intent classification task.

### Cross-Validation Strategy

A **5-fold Stratified Cross-Validation** strategy is used to preserve the original class distribution across all folds, providing a reliable estimate of model performance on unseen data.

### Evaluation Metrics

Each candidate configuration is evaluated using:

- **Accuracy**, which measures overall classification performance.
- **Macro F1-score**, which gives equal importance to every intent class regardless of class frequency, making it more suitable for the Banking77 dataset.

The configuration with the highest **Macro F1-score** is selected as the final model (`refit="f1"`). Training scores are also retained to compare training and validation performance, helping identify potential overfitting before evaluating the selected model on the held-out test set.

In [16]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.model_selection import GridSearchCV

tfidf = TfidfVectorizer()
count = CountVectorizer()

models = {
    "MultinomialNB":{
        "pipeline": Pipeline([
            ("count", count),
            ("clf", MultinomialNB())
        ]),
        "params": {
            "count__ngram_range": [(1, 1), (1, 2), (1, 3)],
            "count__min_df": [1, 2, 5],
            "count__max_df": [0.2, 0.3, 0.4],
            "clf__alpha": [0.01, 0.1, 0.5],
            "clf__fit_prior": [True, False]
        }
    },
    "LogisticRegression": {
        "pipeline": Pipeline([
            ("tfidf", tfidf),
            ("clf", LogisticRegression(max_iter=1000, random_state=42))
        ]),
        "params": {
            "tfidf__min_df": [1, 2, 5],
            "tfidf__max_df": [0.6, 0.7, 0.8],
            "clf__solver": ["lbfgs", "saga"],
            "clf__C": [1, 10, 20]
        }
    },

    "LinearSVC": {
        "pipeline": Pipeline([
            ("tfidf", tfidf),
            ("clf", LinearSVC(max_iter=1000))
        ]),
        "params": {
            "tfidf__min_df": [1, 2, 5],
            "tfidf__max_df": [0.6, 0.7, 0.8],
            "clf__C": [0.5, 1, 2],
            "clf__loss": ["hinge", "squared_hinge"]
        }
    }
}

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import GridSearchCV

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

results = []

for name, model in models.items():

    grid = GridSearchCV(
    estimator=model["pipeline"],
    param_grid=model["params"],
    cv=cv,
    n_jobs=-1,
    scoring={
        "accuracy": "accuracy",
        "f1": "f1_macro"
    },
    refit="f1",
    return_train_score = True
)

    grid.fit(X_train, y_train)

    results.append({
    "Model": name,
    "Vectorizer": model["pipeline"].steps[0][0],
    "Train F1": grid.cv_results_["mean_train_f1"][grid.best_index_],
    "Validation F1": grid.cv_results_["mean_test_f1"][grid.best_index_],
    "Train Accuracy": grid.cv_results_["mean_train_accuracy"][grid.best_index_],
    "Validation Accuracy": grid.cv_results_["mean_test_accuracy"][grid.best_index_],
    "Best Params": grid.best_params_
})

## Model Comparison Results

The cross-validation results show that **Logistic Regression** achieved the best overall performance, obtaining the highest **Macro F1-score (0.86)** and **Accuracy (0.862)** on the validation folds.

Although Logistic Regression produced a considerably higher training score than its validation score, its performance on the validation data remained consistently better than the other candidate models, indicating that it offers the best balance between predictive performance and generalization.

LinearSVC ranked second, delivering competitive results but slightly lower validation performance. Multinomial Naive Bayes achieved the lowest validation scores among the evaluated models, suggesting that its probabilistic assumptions were less suitable for capturing the relationships within the Banking77 dataset.

Based on these results, **Logistic Regression** was selected as the final model and subsequently evaluated on the held-out test set.

In [18]:
results_df = pd.DataFrame(results)
results_df.sort_values("Validation F1", ascending=False)

,Model,Vectorizer,Train F1,Validation F1,Train Accuracy,Validation Accuracy,Best Params
1,LogisticRegression,tfidf,0.972151,0.860083,0.971913,0.862157,"{'clf__C': 10, 'clf__solver': 'lbfgs', 'tfidf__max_df': 0.6, 'tfidf__min_df': 1}"
2,LinearSVC,tfidf,0.951594,0.848336,0.951731,0.852285,"{'clf__C': 1, 'clf__loss': 'squared_hinge', 'tfidf__max_df': 0.6, 'tfidf__min_df': 2}"
0,MultinomialNB,count,0.942379,0.840393,0.943639,0.844912,"{'clf__alpha': 0.1, 'clf__fit_prior': False, 'count__max_df': 0.3, 'count__min_df': 2, 'count__ngram_range': (1, 2)}"


## Save the Best Model

After completing hyperparameter tuning, the best-performing pipeline is saved for future use. Persisting the trained model ensures reproducibility and allows the evaluation notebook to load the exact same model without retraining.

In [ ]:
import joblib

joblib.dump(grid.best_estimator_, "../model/best_model.pkl")

['model/best_model.pkl']